# 📘 Semaine 10 — Communication : Exposés I2C et SPI

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 exposés + 1h30 atelier + 1h30 homework)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs pédagogiques de la semaine

À la fin de cette semaine, l'étudiant sera capable de :

1. **Présenter** de manière claire le bus I2C (2 fils, adressage, ACK/NACK).
2. **Présenter** de manière claire le bus SPI (4 fils, full-duplex, CS).
3. **Comparer** les deux bus (débit, complexité, nombre de fils, applications).
4. **Configurer** I2C1 et SPI1 sur le STM32F103C6T6.
5. **Mettre en œuvre** un scan I2C et un loopback SPI.

---

## 🗺️ Plan de la semaine

| Partie | Contenu | Durée |
|---|---|---|
| **A — Exposés** | Groupes 1 (I2C) et 2 (SPI) + synthèse | 1h30 |
| **B — Atelier** | TP10 : scan I2C + loopback SPI | 1h30 |
| **C — Homework** | Rapport d'exposé + préparation S11 | 1h30 |
| **D — Auto-évaluation** | Checklist finale | 5 min |

---
# 🎓 PARTIE A — EXPOSÉS ÉTUDIANTS (1h30)

## 🔹 Organisation

### Groupes et thématiques

| Groupe | Thème | Durée |
|---|---|---|
| **Groupe 1** | Bus I2C | 20 min + 5 min Q |
| **Groupe 2** | Bus SPI | 20 min + 5 min Q |
| **Enseignant** | Synthèse comparative | 20 min |

### 📋 Grille d'évaluation de l'exposé

| Critère | 0-2 pts |
|---|---|
| Clarté du plan | |
| Exactitude technique | |
| Support visuel | |
| Réponses aux questions | |
| Respect du temps | |
| **Total** | **/10** |

---

## 🔹 Exposé 1 — Bus I2C

### 📝 Plan attendu (groupe 1)

1. **Historique** — inventé par Philips (années 80)
2. **Principe** — bus série synchrone, multi-maître, multi-esclave
3. **Lignes physiques**
   - **SDA** : données
   - **SCL** : horloge
   - **Pull-up** obligatoires (2.2 – 10 kΩ)
4. **Adressage**
   - 7 bits → 112 adresses (16 réservées)
   - 10 bits → 1024 adresses
5. **Protocole**
   - Start (S) : SDA ↓, SCL reste haut
   - Stop (P) : SDA ↑, SCL reste haut
   - ACK / NACK après chaque octet
   - Repeated Start (Sr)
6. **Vitesses**
   - Standard : 100 kHz
   - Fast : 400 kHz
   - Fast+ : 1 MHz
   - High-Speed : 3.4 MHz
7. **Applications** — EEPROM, RTC, capteurs (accéléromètre, température), écran OLED

### 📖 Fiche de référence I2C

#### Signal I2C (exemple : write sur adresse 0x50)

```
SDA :  ──┐     ┌───┬───┬───┬───┬───┬───┬───┬───┬─┐
         └─────┘ A6│A5 │A4 │A3 │A2 │A1 │A0 │W  │A┐
SCL :  ───┐ ┌───┐ ┌───┐ ┌───┐ ┌───┐ ┌───┐ ┌───┐ ┌─┐
          └─┘   └─┘   └─┘   └─┘   └─┘   └─┘   └─┘ 
        START                                    ACK
```

#### Bits Start / Stop

| État | SDA | SCL |
|---|---|---|
| Idle | 1 | 1 |
| START | 1 → 0 | 1 |
| Data | bit | 0/1 |
| STOP | 0 → 1 | 1 |

#### ACK / NACK

| Signal | SDA après le 8ᵉ bit |
|---|---|
| **ACK** | 0 (l'esclave confirme) |
| **NACK** | 1 (pas de confirmation) |

#### Trame complète (écriture)

```
[START] [Adresse 7b + W=0] [ACK] [Adresse registre 8b] [ACK] [Data 8b] [ACK] [STOP]
```

#### Trame complète (lecture)

```
[START] [Adresse 7b + W=0] [ACK] [Adresse registre] [ACK]
[START répété] [Adresse 7b + R=1] [ACK] [Data 8b] [NACK] [STOP]
```

### 🐍 Simulation Python — Trame I2C (10 min)

Simulons une trame I2C complète (écriture puis lecture).

In [ ]:
# ============================================================
# Simulation d'une trame I2C
# ============================================================

def trame_i2c_ecriture(adresse_esclave, registre, data):
    """
    Retourne la trame I2C sous forme de liste de signaux.
    - adresse_esclave : 7 bits
    - registre        : adresse du registre interne (8 bits)
    - data            : donnée à écrire (8 bits)
    """
    trame = []
    trame.append(("START", None))

    # Octet 1 : Adresse 7 bits + bit W = 0
    octet1 = ((adresse_esclave & 0x7F) << 1) | 0
    for i in range(7, -1, -1):
        trame.append((f"ADDR[{i}]", (octet1 >> i) & 1))
    trame.append(("ACK", 0))

    # Octet 2 : Adresse du registre
    for i in range(7, -1, -1):
        trame.append((f"REG[{i}]", (registre >> i) & 1))
    trame.append(("ACK", 0))

    # Octet 3 : Donnée
    for i in range(7, -1, -1):
        trame.append((f"DATA[{i}]", (data >> i) & 1))
    trame.append(("ACK", 0))

    trame.append(("STOP", None))
    return trame

def afficher_trame(trame, titre=""):
    if titre:
        print(f"\n📡 {titre}")
    print("  SDA :", end=" ")
    for nom, val in trame:
        if val is None:
            print(f"[{nom}]", end=" ")
        else:
            print(val, end=" ")
    print()

# Exemple : écrire 0xAB dans le registre 0x10 d'un esclave 0x50
trame = trame_i2c_ecriture(0x50, 0x10, 0xAB)
afficher_trame(trame, "Trame I2C — Écriture")

print(f"\n📊 Nombre total de bits : {sum(1 for _, v in trame if v is not None)}")
print(f"   (8 bits adresse + 1 ACK + 8 bits registre + 1 ACK + 8 bits data + 1 ACK)")
print(f"\n📌 Adresse esclave 0x50 = {0x50:07b}")
print(f"   1er octet envoyé    = 0x{(0x50 << 1):02X} (adresse + W=0)")
print(f"   Pour une lecture    = 0x{(0x50 << 1) | 1:02X} (adresse + R=1)")

### 🐍 Simulation Python — Scan I2C (10 min)

Simulons un scan de bus I2C pour détecter les adresses présentes.

In [ ]:
# ============================================================
# Simulation d'un scan I2C
# ============================================================

def scanner_i2c(appareils_presentes):
    """
    Simule un scan I2C.
    - appareils_presentes : set d'adresses présentes (7 bits)
    Retourne la liste des adresses détectées.
    """
    detectees = []
    for addr in range(0x08, 0x78):   # plage valide 7 bits
        if addr in appareils_presentes:
            detectees.append(addr)
    return detectees

# Appareils typiques sur un bus I2C
APPAREILS = {
    0x3C: "SSD1306 OLED",
    0x48: "ADS1115 ADC",
    0x50: "AT24C32 EEPROM",
    0x68: "DS1307 RTC / MPU6050",
    0x76: "BMP280 Baromètre",
}

print("🔍 Scan I2C — adresses détectées :\n")
print(f"{'Adresse':<12}{'Hex':<8}{'Appareil'}")
print("-" * 45)
for addr in sorted(APPAREILS.keys()):
    print(f"0b{addr:07b}   0x{addr:02X}    {APPAREILS[addr]}")

print(f"\n📊 {len(APPAREILS)} appareil(s) détecté(s) sur le bus")

# Format de sortie type d'un scan (comme sur Arduino/mbed)
print("\n💻 Sortie type d'un scan :")
for addr in sorted(APPAREILS.keys()):
    print(f"   I2C device found at address 0x{addr:02X}  !")

---

## 🔹 Exposé 2 — Bus SPI

### 📝 Plan attendu (groupe 2)

1. **Historique** — développé par Motorola (années 80)
2. **Principe** — bus série synchrone, maître-esclave, full-duplex
3. **Lignes physiques**
   - **MOSI** : Master Out Slave In
   - **MISO** : Master In Slave Out
   - **SCK** : Serial Clock
   - **CS/SS** : Chip Select (actif bas)
4. **Topologie** — 1 maître, N esclaves (CS séparés)
5. **Modes CPOL / CPHA**
   - CPOL : polarité de SCK au repos
   - CPHA : phase d'échantillonnage
   - 4 modes possibles (0, 1, 2, 3)
6. **Vitesses** — jusqu'à 18 MHz sur STM32F103
7. **Applications** — flash externe, écran TFT, ADC/DAC externe, capteurs rapides

### 📖 Fiche de référence SPI

#### Schéma de connexion

```
        ┌─────────────┐                ┌─────────────┐
        │   MAÎTRE    │                │   ESCLAVE   │
        │             │                │             │
        │  MOSI ──────┼────────────────┼──▶ MOSI     │
        │  MISO ◀─────┼────────────────┼─── MISO     │
        │  SCK  ──────┼────────────────┼──▶ SCK      │
        │  CS   ──────┼────────────────┼──▶ CS       │
        └─────────────┘                └─────────────┘
```

#### Modes CPOL / CPHA

| Mode | CPOL | CPHA | Front d'échantillonnage |
|---|---|---|---|
| 0 | 0 | 0 | Front montant |
| 1 | 0 | 1 | Front descendant |
| 2 | 1 | 0 | Front descendant |
| 3 | 1 | 1 | Front montant |

#### Transmission 8 bits

```
CS  : ─┐                                          ┌─
       └──────────────────────────────────────────┘

SCK :  ┌─┐ ┌─┐ ┌─┐ ┌─┐ ┌─┐ ┌─┐ ┌─┐ ┌─┐ ────────
       ┘ └─┘ └─┘ └─┘ └─┘ └─┘ └─┘ └─┘ └────

MOSI:  D7  D6  D5  D4  D3  D2  D1  D0
```

#### Full-duplex

SPI est **full-duplex** : chaque échange envoie **et** reçoit simultanément.
Pour lire, on envoie un octet *dummy* (0x00 ou 0xFF).

### 🐍 Simulation Python — Trame SPI full-duplex (10 min)

In [ ]:
# ============================================================
# Simulation SPI full-duplex
# ============================================================

def spi_echange_octet(octet_mosi, octet_miso):
    """
    Simule un échange SPI : un octet émis, un octet reçu.
    - octet_mosi : octet à envoyer (Master Out)
    - octet_miso : octet envoyé par l'esclave (Master In)
    Retourne (bits_mosi, bits_miso).
    """
    bits_mosi = [(octet_mosi >> i) & 1 for i in range(7, -1, -1)]
    bits_miso = [(octet_miso >> i) & 1 for i in range(7, -1, -1)]
    return bits_mosi, bits_miso

def afficher_spi(bits_mosi, bits_miso, titre=""):
    if titre:
        print(f"\n📡 {titre}")
    print("  Bit    :", " ".join(f"b{i}" for i in range(7, -1, -1)))
    print("  MOSI   :", "  ".join(map(str, bits_mosi)))
    print("  MISO   :", "  ".join(map(str, bits_miso)))

# Exemple : envoyer 0x5A, recevoir 0xA5
mosi, miso = spi_echange_octet(0x5A, 0xA5)
afficher_spi(mosi, miso, "Échange SPI full-duplex")

print(f"\n  Octet envoyé (MOSI) : 0x{0x5A:02X} = 0b{0x5A:08b}")
print(f"  Octet reçu   (MISO) : 0x{0xA5:02X} = 0b{0xA5:08b}")

# Transmission multi-octets
print("\n📊 Échange multi-octets :")
envoi = [0x01, 0x02, 0x03]
recu  = [0xAA, 0xBB, 0xCC]
print(f"  Envoyé : {' '.join(f'0x{v:02X}' for v in envoi)}")
print(f"  Reçu   : {' '.join(f'0x{v:02X}' for v in recu)}")

---

## 🔹 Synthèse comparative (enseignant — 20 min)

### 📊 Tableau comparatif I2C vs SPI

| Critère | I2C | SPI |
|---|---|---|
| **Nombre de fils** | 2 (SDA, SCL) | 4 (MOSI, MISO, SCK, CS) |
| **Vitesse max** | 1 MHz (Fast+) | 18 MHz (STM32F103) |
| **Full-duplex** | ❌ Non (half) | ✅ Oui |
| **Multi-maître** | ✅ Oui (arbitrage) | ❌ Non |
| **Multi-esclave** | ✅ Oui (adressage) | ✅ Oui (CS) |
| **Adressage** | Intégré (7/10 bits) | Aucun (CS sélectionne) |
| **ACK/NACK** | ✅ Oui | ❌ Non |
| **Pull-up** | ✅ Nécessaires | ❌ Non |
| **Distance** | Court (~1 m) | Très court (~30 cm) |
| **Complexité** | Moyenne | Faible |
| **Coût** | Faible (2 fils) | Plus élevé (4 fils + CS) |
| **Applications** | EEPROM, RTC, capteurs lents | Flash, écran, capteurs rapides |

### 📖 Règle de choix

| Besoin | Bus recommandé |
|---|---|
| Peu de fils, plusieurs appareils | **I2C** |
| Débit élevé | **SPI** |
| Communication full-duplex | **SPI** |
| Grand nombre d'esclaves (adressables) | **I2C** |
| Distance moyenne | **I2C** |
| Bus partagé multi-maître | **I2C** |
| Flash / écran graphique | **SPI** |

### 🐍 Comparaison des débits (10 min)

In [ ]:
# ============================================================
# Comparaison des débits I2C vs SPI
# ============================================================

def temps_transfert(taille_octets, frequence_hz, bits_par_octet=8, overhead=0):
    """Calcule le temps de transfert en secondes."""
    total_bits = taille_octets * bits_par_octet + overhead
    return total_bits / frequence_hz

TAILLE = 1024   # 1 Ko à transférer

print(f"⏱️  Temps de transfert de {TAILLE} octets\n")
print(f"{'Bus':<18}{'Vitesse':<14}{'Temps':<15}{'Débit utile'}")
print("-" * 60)

cas = [
    ("I2C Standard",    100_000,    0),       # 100 kHz
    ("I2C Fast",        400_000,    0),       # 400 kHz
    ("I2C Fast+",      1_000_000,   0),       # 1 MHz
    ("SPI 1 MHz",       1_000_000,   0),
    ("SPI 4 MHz",       4_000_000,   0),
    ("SPI 8 MHz",       8_000_000,   0),
    ("SPI 18 MHz",     18_000_000,   0),
]

for nom, f, oh in cas:
    t = temps_transfert(TAILLE, f, overhead=oh)
    debit = TAILLE / t / 1000   # Ko/s
    print(f"{nom:<18}{f/1e6:>6.2f} MHz   {t*1000:>8.3f} ms   {debit:.1f} Ko/s")

print(f"\n📌 À retenir :")
print(f"  → SPI est ~18× plus rapide qu'I2C Standard.")
print(f"  → I2C Fast+ reste 18× plus lent que SPI à 18 MHz.")

---

## 🔹 QCM formatif I2C / SPI (10 min)

**1. Le bus I2C utilise combien de fils ?**  
A. 1  
B. 2  
C. 3  
D. 4

**2. Le signal START en I2C correspond à :**  
A. SDA ↓ pendant que SCL est haut  
B. SDA ↑ pendant que SCL est haut  
C. SCL ↓ pendant que SDA est haut  
D. Les deux lignes tombent ensemble

**3. Le bus SPI est :**  
A. Half-duplex  
B. Full-duplex  
C. Simplex  
D. Multiplex

**4. Combien d'esclaves peut-on connecter en SPI ?**  
A. 1  
B. 8  
C. 112  
D. Autant que de CS disponibles

**5. L'adressage est intégré dans :**  
A. I2C uniquement  
B. SPI uniquement  
C. Les deux  
D. Aucun

**6. La ligne CS du SPI sert à :**  
A. Transmettre les données  
B. Transmettre l'horloge  
C. Sélectionner l'esclave  
D. Alimenter l'esclave

**7. Sur le STM32F103C6T6, I2C1 est sur les broches :**  
A. PA0/PA1  
B. PB6/PB7  
C. PA9/PA10  
D. PA2/PA3

**8. Le mode SPI 0 correspond à :**  
A. CPOL=0, CPHA=0  
B. CPOL=0, CPHA=1  
C. CPOL=1, CPHA=0  
D. CPOL=1, CPHA=1

### ✅ Corrigé du QCM

| Q | Rép. | Justification |
|---|---|---|
| 1 | **B — 2** | SDA + SCL |
| 2 | **A — SDA ↓ SCL haut** | Définition du START |
| 3 | **B — Full-duplex** | MOSI et MISO simultanés |
| 4 | **D — Autant que de CS** | 1 CS par esclave |
| 5 | **A — I2C uniquement** | SPI n'a pas d'adressage |
| 6 | **C — Sélectionner l'esclave** | Chip Select |
| 7 | **B — PB6/PB7** | I2C1 par défaut |
| 8 | **A — CPOL=0, CPHA=0** | Mode 0 |

**Mon score : ___ / 8**

---

# 🛠️ PARTIE B — ATELIER / TP (1h30)

## 🧪 TP10 — Scan I2C + loopback SPI

### 🎯 Objectif
Scanner le bus I2C1 pour détecter des appareils, et tester SPI1 en loopback (MOSI ↔ MISO).

### 📋 Tâches à réaliser (par binôme)

| # | Tâche | Durée | Livrable |
|---|---|---|---|
| 1 | Configurer I2C1 sur PB6 (SCL) et PB7 (SDA) | 15 min | Capture CubeMX |
| 2 | Écrire la fonction scan I2C | 15 min | Code |
| 3 | Connecter un capteur I2C (si dispo) et vérifier l'adresse | 10 min | Démo |
| 4 | Configurer SPI1 sur PA5/PA6/PA7 | 15 min | Capture |
| 5 | Tester SPI en loopback (PA7 ↔ PA6) | 15 min | Démo |
| 6 | Mesurer avec analyseur logique | 15 min | Capture |
| 7 | Rédiger le compte-rendu | 5 min | CR |

### ⚙️ Code — Scan I2C

In [ ]:
/* ============================================================
   TP10 - Scan I2C1 (PB6=SCL, PB7=SDA)
   ============================================================ */

#include "main.h"

I2C_HandleTypeDef hi2c1;
UART_HandleTypeDef huart2;

static void scanner_i2c(void)
{
    char buf[64];
    int n = snprintf(buf, sizeof(buf), "Scan I2C en cours...\r\n");
    HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);

    uint8_t nb_appareils = 0;
    for (uint8_t addr = 0x08; addr < 0x78; addr++)
    {
        // HAL utilise l'adresse décalée à gauche (8 bits)
        if (HAL_I2C_IsDeviceReady(&hi2c1, addr << 1, 3, 10) == HAL_OK)
        {
            n = snprintf(buf, sizeof(buf),
                         "Appareil I2C detecte a l'adresse 0x%02X\r\n", addr);
            HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
            nb_appareils++;
        }
    }

    n = snprintf(buf, sizeof(buf), "Scan termine. %u appareil(s).\r\n", nb_appareils);
    HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_I2C1_Init();
    MX_USART2_UART_Init();

    HAL_Delay(500);
    scanner_i2c();

    while (1)
    {
        HAL_Delay(1000);
    }
}

### ⚙️ Code — Loopback SPI

In [ ]:
/* ============================================================
   TP10 - Loopback SPI1
   - PA5 : SPI1_SCK
   - PA6 : SPI1_MISO
   - PA7 : SPI1_MOSI
   - PA4 : SPI1_NSS (géré en software, non utilisé ici)
   ============================================================ */

#include "main.h"

SPI_HandleTypeDef hspi1;
UART_HandleTypeDef huart2;

static void test_loopback_spi(void)
{
    uint8_t envoi[10]   = {0x01, 0x02, 0x03, 0x04, 0x05,
                           0xAA, 0xBB, 0xCC, 0xDD, 0xEE};
    uint8_t recu[10]    = {0};
    char    buf[80];

    // Transfert SPI (bloque jusqu'à la fin)
    HAL_SPI_TransmitReceive(&hspi1, envoi, recu, 10, 100);

    int n = snprintf(buf, sizeof(buf), "SPI Loopback :\r\n");
    HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);

    for (uint8_t i = 0; i < 10; i++)
    {
        n = snprintf(buf, sizeof(buf),
                     "  [%u] Envoi=0x%02X  Recu=0x%02X  %s\r\n",
                     i, envoi[i], recu[i],
                     (envoi[i] == recu[i]) ? "OK" : "KO");
        HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
    }
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_SPI1_Init();
    MX_USART2_UART_Init();

    HAL_Delay(500);
    test_loopback_spi();

    while (1)
    {
        HAL_Delay(1000);
    }
}

### 🔍 Analyse du code

**Scan I2C :**
| Élément | Rôle |
|---|---|
| `HAL_I2C_IsDeviceReady()` | Teste si un appareil répond à l'adresse |
| `addr << 1` | HAL attend l'adresse 8 bits (7 bits + bit R/W) |
| Plage 0x08..0x77 | Adresses valides 7 bits |

**Loopback SPI :**
| Élément | Rôle |
|---|---|
| `HAL_SPI_TransmitReceive()` | Échange full-duplex |
| Connecter PA6 ↔ PA7 | Boucle MOSI vers MISO |
| `recu[i] == envoi[i]` | Vérification |

### 📖 Configuration CubeMX — I2C1

```
Mode : I2C
I2C Speed Mode : Fast Mode (400 kHz)
Clock Speed : 400000 Hz
Broches : PB6 (I2C1_SCL), PB7 (I2C1_SDA)
```

### 📖 Configuration CubeMX — SPI1

```
Mode : Full-Duplex Master
Hardware NSS : Disabled (NSS Software)
Frame Format : Motorola
Data Size : 8 bits
First Bit : MSB First
Prescaler : 16 (→ ~4.5 MHz à 72 MHz)
Clock Polarity (CPOL) : Low
Clock Phase (CPHA) : 1 Edge
Broches : PA5 (SCK), PA6 (MISO), PA7 (MOSI)
```

### 🐍 Simulation Python — Analyseur logique I2C (15 min)

Simulons un analyseur logique pour visualiser une trame I2C.

In [ ]:
# ============================================================
# Analyseur logique I2C — décodage d'un signal
# ============================================================

def generer_signal_i2c(trame):
    """
    Génère les signaux SDA et SCL à partir d'une trame I2C.
    Chaque élément de trame = (nom, valeur) où valeur peut être None (Start/Stop).
    """
    sda = []
    scl = []
    for nom, val in trame:
        if nom == "START":
            sda += [1, 0]  # descente
            scl += [1, 1]
        elif nom == "STOP":
            sda += [0, 1]  # montée
            scl += [1, 1]
        else:
            # 1 cycle d'horloge par bit : SCL monte au milieu
            sda += [val, val]
            scl += [0, 1]
    return sda, scl

def afficher_logique(sda, scl, largeur=70):
    pas = max(1, len(sda) // largeur)
    ligne_sda = ""
    ligne_scl = ""
    for i in range(0, len(sda), pas):
        bloc_sda = sda[i:i+pas]
        bloc_scl = scl[i:i+pas]
        ligne_sda += "█" if sum(bloc_sda) > len(bloc_sda) / 2 else "_"
        ligne_scl += "█" if sum(bloc_scl) > len(bloc_scl) / 2 else "_"
    print("  SDA :", ligne_sda)
    print("  SCL :", ligne_scl)

# Trame : START, adresse 0x50 + W, registre 0x10, data 0xAB
trame = trame_i2c_ecriture(0x50, 0x10, 0xAB)
sda, scl = generer_signal_i2c(trame)

print("🔬 Analyseur logique — Trame I2C complète\n")
afficher_logique(sda, scl)

print("\n📋 Décodage :")
print("  [START] → [0xA0 = adresse 0x50 + W] → [ACK]")
print("          → [0x10 = registre] → [ACK]")
print("          → [0xAB = data] → [ACK] → [STOP]")

### 🐍 Simulation Python — Comparaison des vitesses I2C/SPI (10 min)

In [ ]:
# ============================================================
# Comparaison visuelle I2C vs SPI
# ============================================================

def comparer_vitesses(taille_octets=64):
    """Compare I2C Standard et SPI 4 MHz."""
    # I2C : 9 bits par octet (8 + ACK) + Start + Stop
    t_i2c = (taille_octets * 9 + 2) / 100_000
    # SPI : 8 bits par octet, pas d'ACK
    t_spi = taille_octets * 8 / 4_000_000
    return t_i2c, t_spi

TAILLE = 64
t_i2c, t_spi = comparer_vitesses(TAILLE)

print(f"📊 Transfert de {TAILLE} octets\n")
print(f"I2C Standard (100 kHz) : {t_i2c*1000:.3f} ms  {'█' * int(t_i2c*10000)}")
print(f"SPI (4 MHz)            : {t_spi*1000:.4f} ms  {'█' * int(t_spi*10000)}")
print(f"\n➡️  SPI est {t_i2c/t_spi:.0f}× plus rapide")

### 📝 Compte-rendu de TP10

**Nom :** __________________  **Prénom :** __________________  **Binôme :** __________________

**1. Configuration CubeMX — I2C1**
- Broches : SCL = ... , SDA = ...
- Vitesse : ... kHz
- Mode : ...

**2. Configuration CubeMX — SPI1**
- Broches : SCK = ... , MISO = ... , MOSI = ...
- Prescaler : ... → F_SPI = ... MHz
- Mode CPOL/CPHA : ...

**3. Scan I2C**
- Nombre d'appareils détectés : ...
- Adresses trouvées : ...
- Appareils identifiés : ...

**4. Test loopback SPI**
- Taille du transfert : ... octets
- Résultat : OK / KO (combien d'octets corrects ?)
- Temps de transfert mesuré : ... µs

**5. Mesures à l'analyseur logique**
- Fréquence SCK mesurée : ... MHz
- Fréquence SCL mesurée : ... kHz
- Observations : ...

**6. Problèmes rencontrés**
- ...

**7. Solutions apportées**
- ...

### 🧪 Exercice bonus — Lecture d'un capteur I2C

Lire un registre d'identification d'un capteur I2C (ex. MPU6050 adresse 0x68, registre WHO_AM_I 0x75).

**Cahier des charges :**
- Lire le registre 0x75 du MPU6050 (attendu : 0x68)
- Envoyer la valeur sur UART
- Gérer les erreurs (device absent)

**Indice :** `HAL_I2C_Mem_Read()`

In [ ]:
// Solution bonus — Lecture WHO_AM_I du MPU6050

static void lire_whoami_mpu6050(void)
{
    uint8_t who_am_i = 0;
    char buf[64];

    HAL_StatusTypeDef status = HAL_I2C_Mem_Read(
        &hi2c1,
        0x68 << 1,    // adresse 8 bits
        0x75,         // registre WHO_AM_I
        I2C_MEMADD_SIZE_8BIT,
        &who_am_i,
        1,
        100
    );

    if (status == HAL_OK) {
        int n = snprintf(buf, sizeof(buf),
                         "MPU6050 WHO_AM_I = 0x%02X\r\n", who_am_i);
        HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
    } else {
        int n = snprintf(buf, sizeof(buf), "MPU6050 non detecte !\r\n");
        HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
    }
}

---

# 🏠 PARTIE C — HOMEWORK (1h30)

## 📚 Exercices à rendre

### 🧩 Exercice 1 — Rapport d'exposé (45 min)

Rédiger un **rapport de 2 pages** sur le thème exposé par ton groupe (I2C ou SPI) :

1. **Introduction** — historique, contexte, intérêt
2. **Description technique** — lignes, protocole, timing
3. **Modes / configurations**
4. **Applications typiques**
5. **Avantages et inconvénients**
6. **Comparaison avec l'autre bus**
7. **Bibliographie** — datasheets, RM0008

**Format :** PDF, 2 pages max, avec schémas.

### 🧩 Exercice 2 — Analyse de trames (30 min)

Décoder les trames I2C suivantes :

**Trame A :**  
```
[START] [0xA0] [ACK] [0x00] [ACK] [0x55] [ACK] [STOP]
```

**Trame B :**  
```
[START] [0xD0] [ACK] [0x75] [ACK]
[Sr]    [0xD1] [ACK] [0x68] [NACK] [STOP]
```

**Questions :**
1. Adresse de l'esclave dans la trame A ?
2. Écriture ou lecture ?
3. Que fait la trame A ?
4. Adresse de l'esclave dans la trame B ?
5. Quelle est l'opération de la trame B ?
6. Que signifie le NACK final ?
7. Que signifie `Sr` ?

In [ ]:
# Corrigé Exercice 2

def decoder_i2c(premier_octet):
    adresse = premier_octet >> 1
    rw = premier_octet & 1
    return adresse, rw

print("📡 Trame A :")
a1, rw1 = decoder_i2c(0xA0)
print(f"  1. Adresse esclave : 0x{a1:02X}")
print(f"  2. Opération       : {'ÉCRITURE' if rw1 == 0 else 'LECTURE'}")
print(f"  3. Écrit 0x55 dans le registre 0x00")

print("\n📡 Trame B :")
b1, rw_b1 = decoder_i2c(0xD0)
b2, rw_b2 = decoder_i2c(0xD1)
print(f"  4. Adresse esclave : 0x{b1:02X}")
print(f"  5. Lecture du registre 0x75")
print(f"  6. NACK final = maître ne demande pas d'octet supplémentaire")
print(f"  7. Sr = Repeated Start (pas de STOP entre les deux phases)")
print(f"\n  Valeur lue : 0x68 (typique pour MPU6050)")

### 🧩 Exercice 3 — Lecture du RM0008 (15 min)

Lire les chapitres **26 (I2C)** et **25 (SPI)** du RM0008 et répondre :

1. Combien d'I2C et de SPI possède le STM32F103C6T6 ?
2. Quel registre configure la vitesse I2C ?
3. Quel bit active le SPI ?
4. Quelle est la différence entre **I2C1** et **I2C2** ?
5. Que signifie le bit **SPE** dans `SPI_CR1` ?
6. Comment activer le mode **Fast Mode** sur I2C ?

### ✍️ Réponses — Exercice 3

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

---

## 🧮 Exercice supplémentaire — Simulation SPI en Python (optionnel)

Implémenter une classe `SPI` qui simule :
- `envoyer_recevoir(octet)` : échange full-duplex
- `configurer(cpol, cpha, prescaler)` : configure les paramètres
- `select(n)` / `deselect(n)` : gère le CS
- Journaliser chaque transaction

In [ ]:
# Corrigé — Simulation SPI

class SPISim:
    def __init__(self):
        self.cpol = 0
        self.cpha = 0
        self.prescaler = 16
        self.esclave_actif = None
        self.journal = []

    def configurer(self, cpol=0, cpha=0, prescaler=16):
        self.cpol = cpol
        self.cpha = cpha
        self.prescaler = prescaler
        self.journal.append(
            f"Config : CPOL={cpol}, CPHA={cpha}, prescaler={prescaler}")

    def select(self, cs):
        self.esclave_actif = cs
        self.journal.append(f"CS{cs} : actif")

    def deselect(self, cs):
        self.journal.append(f"CS{cs} : inactif")
        self.esclave_actif = None

    def envoyer_recevoir(self, octet_emis, octet_recu=0x00):
        if self.esclave_actif is None:
            raise ValueError("Aucun esclave sélectionné")
        self.journal.append(
            f"CS{self.esclave_actif} : 0x{octet_emis:02X} → 0x{octet_recu:02X}")
        return octet_recu

    def afficher_journal(self):
        for ligne in self.journal:
            print(f"  {ligne}")

# Test
spi = SPISim()
spi.configurer(cpol=0, cpha=0, prescaler=16)
spi.select(1)
spi.envoyer_recevoir(0x9F, 0xEF)   # lecture ID flash
spi.envoyer_recevoir(0x00, 0x40)
spi.deselect(1)
spi.afficher_journal()

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 10

Coche ce que tu maîtrises.

- [ ] Je connais les 2 lignes de l'I2C (SDA, SCL) et leur rôle.
- [ ] Je connais les 4 lignes du SPI (MOSI, MISO, SCK, CS).
- [ ] Je sais reconnaître un START et un STOP sur un chronogramme.
- [ ] Je comprends l'adressage I2C (7 bits + R/W).
- [ ] Je connais les 4 modes CPOL/CPHA du SPI.
- [ ] Je sais quelle est la différence de débit entre I2C et SPI.
- [ ] Je sais quand choisir I2C plutôt que SPI (et inversement).
- [ ] J'ai configuré I2C1 en CubeMX.
- [ ] J'ai configuré SPI1 en CubeMX.
- [ ] J'ai implémenté un scan I2C.
- [ ] J'ai testé SPI en loopback.
- [ ] J'ai observé les signaux à l'analyseur logique.
- [ ] J'ai présenté ou préparé mon exposé.
- [ ] J'ai lu les chapitres 25 (SPI) et 26 (I2C) du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour la S11 (UART/USART + SMBus) |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 4 |

---
# 📚 RESSOURCES Semaine 10

### Documents officiels
- 📄 **RM0008** — chapitre 25 (SPI), chapitre 26 (I2C)
- 📄 **Datasheet STM32F103x6** — section 2.3.12 (I2C), 2.3.13 (SPI)
- 📄 **UM1850** — HAL I2C/SPI documentation
- 📄 **AN2586** — STM32F10xxx hardware development

### Documents externes
- 📘 **UM10204** — I2C-bus specification (NXP)
- 📘 **SMBus spec v2.0** (Intel)
- 📘 **Motorola SPI Block Guide**

### Outils
- **STM32CubeMX** — Connectivity → I2C1, SPI1
- **Analyseur logique** (Saleae, PulseView) pour observer les trames
- **Terminal série** pour le scan I2C

### Vidéos
- *I2C vs SPI Explained* — YouTube
- *STM32 I2C Tutorial* — ControllersTech
- *STM32 SPI Tutorial* — ControllersTech

### Bonnes pratiques
- I2C : toujours des **pull-up** sur SDA et SCL (2.2 à 10 kΩ)
- SPI : **CS** bien géré (actif bas) sinon communication instable
- Vérifier les **niveaux de tension** (3.3 V vs 5 V)
- Attention aux **conflits de broches** (PA2/PA3 = USART2)
- Utiliser **HAL_I2C_Mem_Read/Write** pour les capteurs à registres
- En SPI, utiliser **DMA** pour les gros transferts

---

### 🔗 Passage à la semaine 11

**Prochaine séance :** Exposés UART/USART et SMBus  
- UART vs USART (asynchrone vs synchrone)
- Trame série : start, data, parity, stop
- Débit (baud rate) et génération
- SMBus : variante I2C (timeout, PEC, adresses réservées)
- TP11 : UART ↔ PC + démonstration SMBus

**Préparation :**
- Préparer les exposés UART/USART (groupe 3) et SMBus (groupe 4).
- Lire les chapitres 27 (USART) et 26 (I2C/SMBus) du RM0008.

---

**Fin du notebook — Semaine 10** ✨